In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import json
import os
import time

torch.set_num_threads(os.cpu_count())
print(f"torch threads: {torch.get_num_threads()} / cores available: {os.cpu_count()}")

BASE_DIR = Path("..")
PROC_DIR = BASE_DIR / "data" / "processed"

master = pd.read_csv(PROC_DIR / "training_table.csv")

all_cols    = master.columns.tolist()
morgan_cols = [c for c in all_cols if c.startswith('morgan_')]
morpho_cols = [c for c in all_cols if c.startswith('Cells_')
               or c.startswith('Nuclei_')
               or c.startswith('Cytoplasm_')]
ic50_cols   = [c for c in all_cols
               if c not in morgan_cols + morpho_cols + ['drug_name', 'Metadata_JCP2022']]

drug_names    = master['drug_name'].values
drug_index    = {name: i for i, name in enumerate(drug_names)}
morgan_lookup = master.set_index('drug_name')[morgan_cols].values.astype(np.float32)
morpho_lookup = master.set_index('drug_name')[morpho_cols].values.astype(np.float32)

ic50_long = master[['drug_name'] + ic50_cols].melt(
    id_vars='drug_name', var_name='cell_line', value_name='ln_ic50'
).dropna(subset=['ln_ic50']).reset_index(drop=True)

print(f"Drugs:       {len(drug_names)}")
print(f"Morgan cols: {len(morgan_cols)}")
print(f"Morpho cols: {len(morpho_cols)}")
print(f"IC50 pairs:  {len(ic50_long)}")

torch threads: 8 / cores available: 8
Drugs:       175
Morgan cols: 2048
Morpho cols: 3178
IC50 pairs:  149679


In [2]:
# Model classes and training loop

class DrugResponseDataset(Dataset):
    def __init__(self, df, drug_index, morgan_sc, morpho_sc):
        self.drug_idx = torch.tensor(
            [drug_index[d] for d in df['drug_name']], dtype=torch.long)
        self.targets = torch.tensor(df['ln_ic50'].values, dtype=torch.float32)
        self.morgan  = torch.tensor(morgan_sc, dtype=torch.float32)
        self.morpho  = torch.tensor(morpho_sc, dtype=torch.float32)
    def __len__(self): return len(self.targets)
    def __getitem__(self, i):
        idx = self.drug_idx[i]
        return self.morgan[idx], self.morpho[idx], self.targets[i]

class MorganMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2048,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self, morgan, morpho): return self.net(morgan).squeeze(-1)

class MorphoMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3178,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self, morgan, morpho): return self.net(morpho).squeeze(-1)

class ConcatMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5226,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self, morgan, morpho):
        return self.net(torch.cat([morgan, morpho], dim=1)).squeeze(-1)

class CrossAttentionFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178,
                 token_dim=64, num_tokens=16, num_heads=4,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.num_tokens = num_tokens
        self.morgan_token_proj = nn.Linear(morgan_dim // num_tokens, token_dim)
        self.morpho_pad = (num_tokens - (morpho_dim % num_tokens)) % num_tokens
        self.morpho_token_proj = nn.Linear(
            (morpho_dim + self.morpho_pad) // num_tokens, token_dim)
        self.cross_attn_layers = nn.ModuleList([
            nn.MultiheadAttention(token_dim, num_heads,
                                  dropout=dropout, batch_first=True)
            for _ in range(num_layers)])
        self.layer_norms = nn.ModuleList(
            [nn.LayerNorm(token_dim) for _ in range(num_layers)])
        self.readout = nn.Sequential(
            nn.Linear(token_dim*num_tokens*2, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1))
    def tokenize(self, x, proj, n_tokens, pad=0):
        if pad > 0: x = F.pad(x, (0, pad))
        return proj(x.view(x.shape[0], n_tokens, -1))
    def forward(self, morgan, morpho):
        z_m = self.tokenize(morgan, self.morgan_token_proj, self.num_tokens)
        z_p = self.tokenize(morpho, self.morpho_token_proj,
                            self.num_tokens, pad=self.morpho_pad)
        attn_out = z_p
        for attn, norm in zip(self.cross_attn_layers, self.layer_norms):
            attended, _ = attn(attn_out, z_m, z_m)
            attn_out = norm(attn_out + attended)
        fused = torch.cat([attn_out.reshape(attn_out.shape[0], -1),
                           z_m.reshape(z_m.shape[0], -1)], dim=1)
        return self.readout(fused).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178,
                 embed_dim=256, dropout=0.3):
        super().__init__()
        self.morgan_proj = nn.Sequential(
            nn.Linear(morgan_dim, embed_dim), nn.LayerNorm(embed_dim),
            nn.ReLU(), nn.Dropout(dropout))
        self.morpho_proj = nn.Sequential(
            nn.Linear(morpho_dim, embed_dim), nn.LayerNorm(embed_dim),
            nn.ReLU(), nn.Dropout(dropout))
        self.gate = nn.Sequential(
            nn.Linear(embed_dim*2, embed_dim), nn.ReLU(),
            nn.Linear(embed_dim, 2), nn.Softmax(dim=1))
        self.readout = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1))
    def forward(self, morgan, morpho):
        z_m = self.morgan_proj(morgan)
        z_p = self.morpho_proj(morpho)
        gates = self.gate(torch.cat([z_m, z_p], dim=1))
        return self.readout(gates[:,0:1]*z_m + gates[:,1:2]*z_p).squeeze(-1)

def train_model(model, train_loader, val_loader,
                epochs=150, lr=1e-3, patience=20,
                weight_decay=1e-4, clip=False):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    best_val, best_state, patience_ctr = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for mb, mp, mt in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(mb, mp), mt)
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for mb, mp, mt in val_loader:
                val_loss += criterion(model(mb, mp), mt).item() * len(mt)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    return model

print("Model classes and train_model defined. Parameter counts:")
for name, cls in [('MorganMLP', MorganMLP), ('MorphoMLP', MorphoMLP), ('ConcatMLP', ConcatMLP),
                   ('CrossAttentionFusion', CrossAttentionFusion), ('GatedFusion', GatedFusion)]:
    n_params = sum(p.numel() for p in cls().parameters() if p.requires_grad)
    print(f"  {name:22s} {n_params:>9,}")

Model classes and train_model defined. Parameter counts:
  MorganMLP              1,215,233
  MorphoMLP              1,793,793
  ConcatMLP              2,842,369
  CrossAttentionFusion     596,289
  GatedFusion            1,512,835


In [3]:
# Corrected evaluation: scores at drug level (n=26 test drugs) instead of pair level,
# matching RF/GBM/Ridge

def evaluate_both(model, loader, test_df):
    """
    loader must be built with shuffle=False, and test_df must be the exact
    DataFrame used to build that loader — prediction order must match test_df
    row order to attach drug names back to predictions correctly.
    """
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for mb, mp, mt in loader:
            preds.extend(model(mb, mp).numpy())
            targets.extend(mt.numpy())
    preds, targets = np.array(preds), np.array(targets)

    r_pair, _ = pearsonr(preds, targets)
    rmse_pair = np.sqrt(((preds - targets) ** 2).mean())

    out = test_df.copy()
    out['pred'] = preds

    within_drug_std = out.groupby('drug_name')['pred'].std().max()
    assert within_drug_std < 1e-4, (
        f"prediction varies within a drug (max std={within_drug_std:.6f}) — "
        f"loader shuffle is probably True, or test_df doesn't match loader order"
    )

    drug_level = out.groupby('drug_name').agg(pred=('pred', 'first'), true=('ln_ic50', 'mean'))
    r_drug, _ = pearsonr(drug_level['pred'], drug_level['true'])
    rmse_drug = np.sqrt(((drug_level['pred'] - drug_level['true']) ** 2).mean())

    return r_pair, rmse_pair, r_drug, rmse_drug, drug_level

print("evaluate_both defined.")

evaluate_both defined.


In [4]:
# Stage 2: full 25-seed x 5-condition rerun with drug-level scoring.
# Resumable by construction, before training each model, checks whether its
# prediction CSV already exists and loads the score instead of retraining.
# Safe to rerun this cell unmodified after any interruption, at any point.

FULL_SEEDS = list(range(25))
CONDITIONS = [
    ('structure_only',   MorganMLP),
    ('morphology_only',  MorphoMLP),
    ('concatenation',    ConcatMLP),
    ('cross_attention',  CrossAttentionFusion),
    ('gated_fusion',     GatedFusion),
]

PRED_DIR = BASE_DIR / "results" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = BASE_DIR / "results" / "multiseed_25_results_drugscored_CHECKPOINT.json"

full_results = {name: {'r_pair': [], 'r_drug': [], 'rmse_pair': [], 'rmse_drug': []}
                 for name, _ in CONDITIONS}

t_start = time.time()
n_trained, n_loaded = 0, 0

for seed in FULL_SEEDS:
    train_loader = val_loader = test_loader = test_df = None  # built lazily, only if this seed needs training

    for name, ModelClass in CONDITIONS:
        csv_path = PRED_DIR / f"drug_level_{name}_seed{seed}.csv"

        if csv_path.exists():
            dl = pd.read_csv(csv_path, index_col=0)
            r_d, _ = pearsonr(dl['pred'], dl['true'])
            rmse_d = np.sqrt(((dl['pred'] - dl['true']) ** 2).mean())
            full_results[name]['r_drug'].append(float(r_d))
            full_results[name]['rmse_drug'].append(float(rmse_d))
            full_results[name]['r_pair'].append(None)    # not stored on disk; diagnostic-only, unused in Table 2
            full_results[name]['rmse_pair'].append(None)
            n_loaded += 1
            elapsed = (time.time() - t_start) / 60
            print(f"[{elapsed:6.1f} min] seed={seed:2d} {name:16s} | loaded from disk -> drug R={r_d:.3f} RMSE={rmse_d:.3f}")
            continue

        if train_loader is None:
            torch.manual_seed(seed)
            np.random.seed(seed)
            shuffled_idx = np.random.permutation(len(drug_names))
            n_test  = int(len(drug_names) * 0.15)
            n_val   = int(len(drug_names) * 0.15)
            n_train = len(drug_names) - n_test - n_val
            train_drugs = set(drug_names[shuffled_idx[:n_train]])
            val_drugs   = set(drug_names[shuffled_idx[n_train:n_train+n_val]])
            test_drugs  = set(drug_names[shuffled_idx[n_train+n_val:]])

            train_df = ic50_long[ic50_long['drug_name'].isin(train_drugs)].reset_index(drop=True)
            val_df   = ic50_long[ic50_long['drug_name'].isin(val_drugs)].reset_index(drop=True)
            test_df  = ic50_long[ic50_long['drug_name'].isin(test_drugs)].reset_index(drop=True)

            train_idx     = [drug_index[d] for d in train_drugs]
            morpho_scaler = StandardScaler()
            morpho_sc     = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)
            morgan_sc     = morgan_lookup

            train_ds = DrugResponseDataset(train_df, drug_index, morgan_sc, morpho_sc)
            val_ds   = DrugResponseDataset(val_df,   drug_index, morgan_sc, morpho_sc)
            test_ds  = DrugResponseDataset(test_df,  drug_index, morgan_sc, morpho_sc)
            train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
            val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)
            test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

        model = train_model(ModelClass(), train_loader, val_loader)
        r_p, rmse_p, r_d, rmse_d, drug_level = evaluate_both(model, test_loader, test_df)
        full_results[name]['r_pair'].append(float(r_p))
        full_results[name]['r_drug'].append(float(r_d))
        full_results[name]['rmse_pair'].append(float(rmse_p))
        full_results[name]['rmse_drug'].append(float(rmse_d))
        drug_level.to_csv(csv_path)
        n_trained += 1
        elapsed = (time.time() - t_start) / 60
        print(f"[{elapsed:6.1f} min] seed={seed:2d} {name:16s} | "
              f"pair R={r_p:.3f}  ->  drug R={r_d:.3f}  RMSE={rmse_d:.3f}")

    with open(CKPT_PATH, "w") as f:
        json.dump({'completed_through_seed': seed, 'results': full_results}, f, indent=2)

print("\n" + "="*70)
print(f"Stage 2 complete. Trained {n_trained} new models, loaded {n_loaded} from disk. Total elapsed: {(time.time()-t_start)/60:.1f} min")
print("="*70)
for name in full_results:
    rd = np.array(full_results[name]['r_drug']); ed = np.array(full_results[name]['rmse_drug'])
    print(f"{name:16s} | n={len(rd)}  drug R={rd.mean():.3f}±{rd.std(ddof=1):.3f}  RMSE={ed.mean():.3f}±{ed.std(ddof=1):.3f}")

with open(BASE_DIR / "results" / "multiseed_25_results_drugscored.json", "w") as f:
    json.dump(full_results, f, indent=2)
print("\nSaved: multiseed_25_results_drugscored.json")

[   0.0 min] seed= 0 structure_only   | loaded from disk -> drug R=0.131 RMSE=2.124
[   0.0 min] seed= 0 morphology_only  | loaded from disk -> drug R=0.808 RMSE=1.105
[   0.0 min] seed= 0 concatenation    | loaded from disk -> drug R=0.815 RMSE=1.071
[   0.0 min] seed= 0 cross_attention  | loaded from disk -> drug R=0.871 RMSE=0.901
[   0.0 min] seed= 0 gated_fusion     | loaded from disk -> drug R=0.510 RMSE=1.602
[   0.0 min] seed= 1 structure_only   | loaded from disk -> drug R=0.490 RMSE=1.609
[   0.0 min] seed= 1 morphology_only  | loaded from disk -> drug R=0.624 RMSE=1.277
[   0.0 min] seed= 1 concatenation    | loaded from disk -> drug R=0.647 RMSE=1.240
[   0.0 min] seed= 1 cross_attention  | loaded from disk -> drug R=0.681 RMSE=1.222
[   0.0 min] seed= 1 gated_fusion     | loaded from disk -> drug R=0.721 RMSE=1.132
[   0.0 min] seed= 2 structure_only   | loaded from disk -> drug R=0.539 RMSE=1.962
[   0.0 min] seed= 2 morphology_only  | loaded from disk -> drug R=0.848 RMS

In [5]:
# RF and GBM on all three feature sets, 25 seeds

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

drug_mean_ic50 = ic50_long.groupby('drug_name')['ln_ic50'].mean()

rf_gbm_results = {
    f'{model}_{feat}': {'r': [], 'rmse': []}
    for model in ['rf', 'gbm']
    for feat in ['structure_only', 'morphology_only', 'concatenation']
}

t_start = time.time()

for seed in range(25):
    np.random.seed(seed)
    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val
    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    test_drugs  = set(drug_names[shuffled_idx[n_train + n_val:]])
    train_idx = [drug_index[d] for d in train_drugs]
    test_idx  = [drug_index[d] for d in test_drugs]

    y_train = drug_mean_ic50.loc[list(train_drugs)].values
    y_test  = drug_mean_ic50.loc[list(test_drugs)].values

    morpho_scaler = StandardScaler()
    morpho_sc = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)

    feature_sets = {
        'structure_only':  (morgan_lookup[train_idx], morgan_lookup[test_idx]),
        'morphology_only': (morpho_sc[train_idx],      morpho_sc[test_idx]),
        'concatenation':   (np.hstack([morgan_lookup[train_idx], morpho_sc[train_idx]]),
                             np.hstack([morgan_lookup[test_idx],  morpho_sc[test_idx]])),
    }

    for feat_name, (X_train, X_test) in feature_sets.items():
        rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=seed)
        rf.fit(X_train, y_train)
        rf_pred = rf.predict(X_test)
        r_rf, _ = pearsonr(rf_pred, y_test)
        rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))
        rf_gbm_results[f'rf_{feat_name}']['r'].append(float(r_rf))
        rf_gbm_results[f'rf_{feat_name}']['rmse'].append(float(rmse_rf))

        gbm = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=seed)
        gbm.fit(X_train, y_train)
        gbm_pred = gbm.predict(X_test)
        r_gbm, _ = pearsonr(gbm_pred, y_test)
        rmse_gbm = np.sqrt(mean_squared_error(y_test, gbm_pred))
        rf_gbm_results[f'gbm_{feat_name}']['r'].append(float(r_gbm))
        rf_gbm_results[f'gbm_{feat_name}']['rmse'].append(float(rmse_gbm))

    if seed % 5 == 0 or seed == 24:
        print(f"[{(time.time()-t_start)/60:5.1f} min] seed={seed:2d} done")

print("\n" + "="*70)
print("RF/GBM: all three feature sets (mean ± std, n=25 seeds)")
print("="*70)
for key in rf_gbm_results:
    r = np.array(rf_gbm_results[key]['r']); e = np.array(rf_gbm_results[key]['rmse'])
    print(f"{key:26s} | R={r.mean():.3f}±{r.std(ddof=1):.3f}  RMSE={e.mean():.3f}±{e.std(ddof=1):.3f}")

with open(BASE_DIR / "results" / "rf_gbm_25seed_all_features.json", "w") as f:
    json.dump(rf_gbm_results, f, indent=2)
print("\nSaved: rf_gbm_25seed_all_features.json")

[  2.3 min] seed= 0 done
[ 12.8 min] seed= 5 done
[ 23.3 min] seed=10 done
[ 33.1 min] seed=15 done
[ 42.6 min] seed=20 done
[ 50.3 min] seed=24 done

RF/GBM: all three feature sets (mean ± std, n=25 seeds)
rf_structure_only          | R=0.409±0.210  RMSE=1.785±0.309
rf_morphology_only         | R=0.764±0.112  RMSE=1.273±0.302
rf_concatenation           | R=0.776±0.101  RMSE=1.247±0.288
gbm_structure_only         | R=0.372±0.223  RMSE=1.831±0.286
gbm_morphology_only        | R=0.753±0.088  RMSE=1.327±0.260
gbm_concatenation          | R=0.767±0.095  RMSE=1.276±0.253

Saved: rf_gbm_25seed_all_features.json


In [6]:
# RidgeCV on all three feature sets, 25 seeds

from sklearn.linear_model import RidgeCV

ALPHAS = np.logspace(-3, 5, 25)

ridge_results = {feat: {'r': [], 'rmse': [], 'best_alpha': []}
                  for feat in ['structure_only', 'morphology_only', 'concatenation']}

t_start = time.time()

for seed in range(25):
    np.random.seed(seed)
    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val
    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    test_drugs  = set(drug_names[shuffled_idx[n_train + n_val:]])
    train_idx = [drug_index[d] for d in train_drugs]
    test_idx  = [drug_index[d] for d in test_drugs]

    y_train = drug_mean_ic50.loc[list(train_drugs)].values
    y_test  = drug_mean_ic50.loc[list(test_drugs)].values

    morpho_scaler = StandardScaler()
    morpho_sc = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)
    morgan_scaler = StandardScaler()
    morgan_sc = morgan_scaler.fit(morgan_lookup[train_idx]).transform(morgan_lookup)  # actually scaled this time

    feature_sets = {
        'structure_only':  (morgan_sc[train_idx], morgan_sc[test_idx]),
        'morphology_only': (morpho_sc[train_idx],  morpho_sc[test_idx]),
        'concatenation':   (np.hstack([morgan_sc[train_idx], morpho_sc[train_idx]]),
                             np.hstack([morgan_sc[test_idx],  morpho_sc[test_idx]])),
    }

    for feat_name, (X_train, X_test) in feature_sets.items():
        ridge = RidgeCV(alphas=ALPHAS, cv=5)
        ridge.fit(X_train, y_train)
        y_pred = ridge.predict(X_test)
        r, _ = pearsonr(y_pred, y_test)
        rmse = np.sqrt(((y_pred - y_test) ** 2).mean())
        ridge_results[feat_name]['r'].append(float(r))
        ridge_results[feat_name]['rmse'].append(float(rmse))
        ridge_results[feat_name]['best_alpha'].append(float(ridge.alpha_))

    if seed % 5 == 0 or seed == 24:
        print(f"[{(time.time()-t_start)/60:5.1f} min] seed={seed:2d} done")

print("\n" + "="*70)
print("RidgeCV — all three feature sets (mean ± std, n=25 seeds)")
print("="*70)
for feat in ridge_results:
    r = np.array(ridge_results[feat]['r']); e = np.array(ridge_results[feat]['rmse'])
    a = np.array(ridge_results[feat]['best_alpha'])
    print(f"{feat:18s} | R={r.mean():.3f}±{r.std(ddof=1):.3f}  RMSE={e.mean():.3f}±{e.std(ddof=1):.3f}  median alpha={np.median(a):.2f}")

with open(BASE_DIR / "results" / "ridgecv_25seed_all_features.json", "w") as f:
    json.dump(ridge_results, f, indent=2)
print("\nSaved: ridgecv_25seed_all_features.json")

C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:250: LinAlgWarning: Ill-conditioned matrix (rcond=4.02194e-09): result may not be accurate.
  dual_coef = lin

[  0.0 min] seed= 0 done


C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:250: LinAlgWarning: Ill-conditioned matrix (rcond=4.79033e-09): result may not be accurate.
  dual_coef = lin

[  0.3 min] seed= 5 done


C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:250: LinAlgWarning: Ill-conditioned matrix (rcond=3.15337e-09): result may not be accurate.
  dual_coef = lin

...

[  0.9 min] seed=15 done


C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:250: LinAlgWarning: Ill-conditioned matrix (rcond=3.50322e-09): result may not be accurate.
  dual_coef = lin

[  1.1 min] seed=20 done


C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:252: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
C:\ProgramData\miniconda3\envs\jumpcp\lib\site-packages\sklearn\linear_model\_ridge.py:250: LinAlgWarning: Ill-conditioned matrix (rcond=3.76082e-09): result may not be accurate.
  dual_coef = lin

[  1.3 min] seed=24 done

RidgeCV — all three feature sets (mean ± std, n=25 seeds)
structure_only     | R=0.333±0.251  RMSE=1.831±0.336  median alpha=46.42
morphology_only    | R=0.731±0.157  RMSE=1.335±0.370  median alpha=4641.59
concatenation      | R=0.730±0.145  RMSE=1.345±0.374  median alpha=4641.59

Saved: ridgecv_25seed_all_features.json


In [9]:
# Ablation: do MLPs trained on the SAME clean, pre-averaged drug-level target
# RF/GBM/Ridge use close any of the gap?

class DrugLevelDataset(Dataset):
    def __init__(self, drugs, drug_index, morgan_sc, morpho_sc, y):
        self.drug_idx = torch.tensor([drug_index[d] for d in drugs], dtype=torch.long)
        self.targets  = torch.tensor(y, dtype=torch.float32)
        self.morgan   = torch.tensor(morgan_sc, dtype=torch.float32)
        self.morpho   = torch.tensor(morpho_sc, dtype=torch.float32)
    def __len__(self): return len(self.targets)
    def __getitem__(self, i):
        idx = self.drug_idx[i]
        return self.morgan[idx], self.morpho[idx], self.targets[i]

ablation_results = {name: {'r_drug': [], 'rmse_drug': []}
                     for name in ['structure_only', 'morphology_only', 'concatenation']}

t_start = time.time()

for seed in range(25):
    torch.manual_seed(seed)
    np.random.seed(seed)
    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val
    train_drugs = list(drug_names[shuffled_idx[:n_train]])
    val_drugs   = list(drug_names[shuffled_idx[n_train:n_train+n_val]])
    test_drugs  = list(drug_names[shuffled_idx[n_train+n_val:]])

    train_idx = [drug_index[d] for d in train_drugs]
    morpho_scaler = StandardScaler()
    morpho_sc = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)
    morgan_sc = morgan_lookup

    y_train = drug_mean_ic50.loc[train_drugs].values
    y_val   = drug_mean_ic50.loc[val_drugs].values
    y_test  = drug_mean_ic50.loc[test_drugs].values

    train_loader = DataLoader(DrugLevelDataset(train_drugs, drug_index, morgan_sc, morpho_sc, y_train), batch_size=32, shuffle=True)
    val_loader   = DataLoader(DrugLevelDataset(val_drugs,   drug_index, morgan_sc, morpho_sc, y_val),   batch_size=32, shuffle=False)
    test_loader  = DataLoader(DrugLevelDataset(test_drugs,  drug_index, morgan_sc, morpho_sc, y_test),  batch_size=32, shuffle=False)

    for name, ModelClass in [('structure_only', MorganMLP), ('morphology_only', MorphoMLP), ('concatenation', ConcatMLP)]:
        model = train_model(ModelClass(), train_loader, val_loader)
        model.eval()
        preds = []
        with torch.no_grad():
            for mb, mp, mt in test_loader:
                preds.extend(model(mb, mp).numpy())
        preds = np.array(preds)
        r, _ = pearsonr(preds, y_test)
        rmse = np.sqrt(((preds - y_test) ** 2).mean())
        ablation_results[name]['r_drug'].append(float(r))
        ablation_results[name]['rmse_drug'].append(float(rmse))

    if seed % 5 == 0 or seed == 24:
        print(f"[{(time.time()-t_start)/60:5.1f} min] seed={seed:2d} done")

print("\n" + "="*70)
print("MLPs trained directly on drug-level target (n=123 rows), n=25 seeds")
print("="*70)
for name in ablation_results:
    r = np.array(ablation_results[name]['r_drug']); e = np.array(ablation_results[name]['rmse_drug'])
    print(f"{name:16s} | R={r.mean():.3f}±{r.std(ddof=1):.3f}  RMSE={e.mean():.3f}±{e.std(ddof=1):.3f}")
print("\nFor reference: [RF-concatenation: R=0.776 RMSE=1.247]")
print("Original per-cell-line-trained MLPs: structure R=0.351, morphology R=0.701, concatenation R=0.720")

with open(BASE_DIR / "results" / "mlp_druglevel_target_ablation.json", "w") as f:
    json.dump(ablation_results, f, indent=2)
print("\nSaved: mlp_druglevel_target_ablation.json")

[  0.3 min] seed= 0 done
[  2.0 min] seed= 5 done
[  3.7 min] seed=10 done
[  5.5 min] seed=15 done
[  7.7 min] seed=20 done
[  9.0 min] seed=24 done

MLPs trained directly on drug-level target (n=123 rows), n=25 seeds
structure_only   | R=0.404±0.215  RMSE=1.857±0.324
morphology_only  | R=0.721±0.114  RMSE=1.401±0.260
concatenation    | R=0.733±0.136  RMSE=1.371±0.296

For reference: [RF-concatenation: R=0.776 RMSE=1.247]
Original per-cell-line-trained MLPs: structure R=0.351, morphology R=0.701, concatenation R=0.720

Saved: mlp_druglevel_target_ablation.json


In [7]:
# Exact SHAP attribution on RF-concatenation, the best performing model overall (R=0.776).


import shap
from sklearn.ensemble import RandomForestRegressor

n_morgan = len(morgan_cols)
n_morpho = len(morpho_cols)

shap_summary = {'pct_morphology': [], 'pct_structure': [],
                 'morgan_mean_abs_active': [], 'morpho_mean_abs': []}

t_start = time.time()

for seed in range(25):
    np.random.seed(seed)
    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val
    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    test_drugs  = set(drug_names[shuffled_idx[n_train + n_val:]])
    train_idx = [drug_index[d] for d in train_drugs]
    test_idx  = [drug_index[d] for d in test_drugs]

    y_train = drug_mean_ic50.loc[list(train_drugs)].values

    morpho_scaler = StandardScaler()
    morpho_sc = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)

    X_train = np.hstack([morgan_lookup[train_idx], morpho_sc[train_idx]])
    X_test  = np.hstack([morgan_lookup[test_idx],  morpho_sc[test_idx]])

    rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=seed)
    rf.fit(X_train, y_train)

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)  # exact, (n_test, n_morgan+n_morpho)

    morgan_shap = shap_values[:, :n_morgan]
    morpho_shap = shap_values[:, n_morgan:]

    total_morgan = np.abs(morgan_shap).sum()
    total_morpho = np.abs(morpho_shap).sum()
    pct_morpho = 100 * total_morpho / (total_morgan + total_morpho)

    active_mask = X_test[:, :n_morgan] != 0
    morgan_active_mean = np.abs(morgan_shap)[active_mask].mean() if active_mask.sum() > 0 else np.nan
    morpho_mean = np.abs(morpho_shap).mean()

    shap_summary['pct_morphology'].append(float(pct_morpho))
    shap_summary['pct_structure'].append(float(100 - pct_morpho))
    shap_summary['morgan_mean_abs_active'].append(float(morgan_active_mean))
    shap_summary['morpho_mean_abs'].append(float(morpho_mean))

    if seed % 5 == 0 or seed == 24:
        print(f"[{(time.time()-t_start)/60:5.1f} min] seed={seed:2d} | morphology={pct_morpho:.1f}%  structure={100-pct_morpho:.1f}%")

pct = np.array(shap_summary['pct_morphology'])
print("\n" + "="*60)
print(f"RF-concatenation SHAP attribution, n=25 seeds (exact, TreeExplainer)")
print("="*60)
print(f"Morphology: {pct.mean():.1f}% ± {pct.std(ddof=1):.1f}%  (range {pct.min():.1f}–{pct.max():.1f}%)")
print(f"Structure:  {100-pct.mean():.1f}% ± {pct.std(ddof=1):.1f}%")
print(f"\nDensity-adjusted (mean |SHAP| per feature):")
print(f"  Morgan (active bits only): {np.mean(shap_summary['morgan_mean_abs_active']):.5f}")
print(f"  Morphology (all features): {np.mean(shap_summary['morpho_mean_abs']):.5f}")

with open(BASE_DIR / "results" / "shap_rf_concat_25seed.json", "w") as f:
    json.dump(shap_summary, f, indent=2)
print("\nSaved: shap_rf_concat_25seed.json")

[  0.2 min] seed= 0 | morphology=95.2%  structure=4.8%
[  1.4 min] seed= 5 | morphology=93.5%  structure=6.5%
[  2.6 min] seed=10 | morphology=93.8%  structure=6.2%
[  3.8 min] seed=15 | morphology=94.4%  structure=5.6%
[  4.9 min] seed=20 | morphology=95.4%  structure=4.6%
[  5.8 min] seed=24 | morphology=93.9%  structure=6.1%

RF-concatenation SHAP attribution, n=25 seeds (exact, TreeExplainer)
Morphology: 95.0% ± 1.4%  (range 92.5–98.8%)
Structure:  5.0% ± 1.4%

Density-adjusted (mean |SHAP| per feature):
  Morgan (active bits only): 0.00153
  Morphology (all features): 0.00084

Saved: shap_rf_concat_25seed.json


In [3]:
# Corrected statistics: paired t-tests and 95% CIs on drug-level R,
# computed directly from the saved 25-seed results.

from scipy.stats import ttest_rel
from scipy.stats import t as tdist

with open(BASE_DIR / "results" / "multiseed_25_results_drugscored.json") as f:
    mlp_results = json.load(f)

conditions = ['structure_only', 'morphology_only', 'concatenation', 'cross_attention', 'gated_fusion']
r_drug = {c: np.array(mlp_results[c]['r_drug']) for c in conditions}

print("Mean +/- SD, drug-level R (n=25 seeds)")
for c in conditions:
    print(f"  {c:16s} R={r_drug[c].mean():.3f} +/- {r_drug[c].std(ddof=1):.3f}")

print("\n95% CI on mean R (t-distribution, df=24)")
ci_results = {}
for c in conditions:
    arr = r_drug[c]
    n = len(arr)
    se = arr.std(ddof=1) / np.sqrt(n)
    tcrit = tdist.ppf(0.975, df=n-1)
    lo, hi = arr.mean() - tcrit*se, arr.mean() + tcrit*se
    ci_results[c] = {'mean': float(arr.mean()), 'ci_low': float(lo), 'ci_high': float(hi)}
    print(f"  {c:16s} R={arr.mean():.3f}  95% CI [{lo:.3f}, {hi:.3f}]")

print("\nPaired t-tests (matched seeds)")
comparisons = [
    ('structure_only', 'morphology_only'),
    ('structure_only', 'concatenation'),
    ('morphology_only','concatenation'),
    ('concatenation',  'cross_attention'),
    ('concatenation',  'gated_fusion'),
]
ttest_results = {}
for a, b in comparisons:
    t, p = ttest_rel(r_drug[b], r_drug[a])
    dR = r_drug[b].mean() - r_drug[a].mean()
    key = f"{b}_vs_{a}"
    ttest_results[key] = {'t': float(t), 'p': float(p), 'delta_R': float(dR)}
    print(f"  {b} vs {a}: t={t:.2f} p={p:.4g} deltaR={dR:+.3f}")

with open(BASE_DIR / "results" / "ablation_stats_corrected.json", "w") as f:
    json.dump({'ci': ci_results, 'paired_tests': ttest_results}, f, indent=2)
print("\nSaved: ablation_stats_corrected.json")

Mean +/- SD, drug-level R (n=25 seeds)
  structure_only   R=0.351 +/- 0.221
  morphology_only  R=0.701 +/- 0.176
  concatenation    R=0.720 +/- 0.135
  cross_attention  R=0.728 +/- 0.128
  gated_fusion     R=0.645 +/- 0.148

95% CI on mean R (t-distribution, df=24)
  structure_only   R=0.351  95% CI [0.260, 0.443]
  morphology_only  R=0.701  95% CI [0.628, 0.773]
  concatenation    R=0.720  95% CI [0.664, 0.775]
  cross_attention  R=0.728  95% CI [0.676, 0.781]
  gated_fusion     R=0.645  95% CI [0.584, 0.705]

Paired t-tests (matched seeds)
  morphology_only vs structure_only: t=6.25 p=1.843e-06 deltaR=+0.349
  concatenation vs structure_only: t=7.45 p=1.088e-07 deltaR=+0.368
  concatenation vs morphology_only: t=1.34 p=0.1925 deltaR=+0.019
  cross_attention vs concatenation: t=0.70 p=0.493 deltaR=+0.009
  gated_fusion vs concatenation: t=-3.27 p=0.00323 deltaR=-0.075

Saved: ablation_stats_corrected.json


In [6]:
# Formal paired tests: classical ML vs MLP. All models trained on identical
# seed-generated splits, so paired tests are valid throughout.

with open(BASE_DIR / "results" / "rf_gbm_25seed_all_features.json") as f:
    rf_gbm = json.load(f)
with open(BASE_DIR / "results" / "ridgecv_25seed_all_features.json") as f:
    ridge = json.load(f)

rf_concat    = np.array(rf_gbm['rf_concatenation']['r'])
rf_struct    = np.array(rf_gbm['rf_structure_only']['r'])
ridge_concat = np.array(ridge['concatenation']['r'])

classical_comparisons = [
    ('RF-concatenation vs MLP-concatenation',              rf_concat,    r_drug['concatenation']),
    ('RF-concatenation vs MLP-cross_attention (best neural)', rf_concat, r_drug['cross_attention']),
    ('RF-structure vs MLP-structure',                       rf_struct,    r_drug['structure_only']),
    ('RidgeCV-concatenation vs MLP-concatenation',          ridge_concat, r_drug['concatenation']),
]

print("Paired t-tests: classical ML vs MLP")
classical_ttest_results = {}
for label, classical_arr, mlp_arr in classical_comparisons:
    t, p = ttest_rel(classical_arr, mlp_arr)
    dR = classical_arr.mean() - mlp_arr.mean()
    classical_ttest_results[label] = {'t': float(t), 'p': float(p), 'delta_R': float(dR)}
    print(f"  {label}: t={t:.2f} p={p:.4g} deltaR={dR:+.3f}")

with open(BASE_DIR / "results" / "classical_vs_mlp_stats.json", "w") as f:
    json.dump(classical_ttest_results, f, indent=2)
print("\nSaved: classical_vs_mlp_stats.json")

Paired t-tests: classical ML vs MLP
  RF-concatenation vs MLP-concatenation: t=4.36 p=0.0002113 deltaR=+0.056
  RF-concatenation vs MLP-cross_attention (best neural): t=4.40 p=0.0001899 deltaR=+0.048
  RF-structure vs MLP-structure: t=2.52 p=0.01863 deltaR=+0.058
  RidgeCV-concatenation vs MLP-concatenation: t=0.88 p=0.3856 deltaR=+0.011

Saved: classical_vs_mlp_stats.json


In [9]:
# Pathway-level SHAP attribution on RF-concatenation, pooled across 25 seeds,
# replaces the original single-seed MLP pathway breakdown.

from sklearn.ensemble import RandomForestRegressor
import shap
from collections import defaultdict

n_morgan = len(morgan_cols)
n_morpho = len(morpho_cols)
drug_mean_ic50 = ic50_long.groupby('drug_name')['ln_ic50'].mean()

gdsc2_matched = pd.read_csv(PROC_DIR / "gdsc2_matched.csv")
DRUG_COL = "DRUG_NAME"
PATHWAY_COL = "PATHWAY_NAME"
drug_to_pathway = gdsc2_matched.drop_duplicates(DRUG_COL).set_index(DRUG_COL)[PATHWAY_COL].to_dict()

missing = [d for d in drug_names if d not in drug_to_pathway]
print(f"Drugs with no pathway match: {len(missing)} of {len(drug_names)}")

pathway_morpho_shap = defaultdict(list)
pathway_struct_shap = defaultdict(list)

t_start = time.time()

for seed in range(25):
    np.random.seed(seed)
    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val
    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    test_drugs  = list(drug_names[shuffled_idx[n_train + n_val:]])
    train_idx = [drug_index[d] for d in train_drugs]
    test_idx  = [drug_index[d] for d in test_drugs]

    y_train = drug_mean_ic50.loc[list(train_drugs)].values
    morpho_scaler = StandardScaler()
    morpho_sc = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)

    X_train = np.hstack([morgan_lookup[train_idx], morpho_sc[train_idx]])
    X_test  = np.hstack([morgan_lookup[test_idx],  morpho_sc[test_idx]])

    rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=seed)
    rf.fit(X_train, y_train)

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)
    morgan_shap = np.abs(shap_values[:, :n_morgan])
    morpho_shap = np.abs(shap_values[:, n_morgan:])

    for i, drug in enumerate(test_drugs):
        pathway = drug_to_pathway.get(drug)
        if pathway is None:
            continue
        pathway_morpho_shap[pathway].append(morpho_shap[i].sum())
        pathway_struct_shap[pathway].append(morgan_shap[i].sum())

    if seed % 5 == 0 or seed == 24:
        print(f"[{(time.time()-t_start)/60:5.1f} min] seed={seed:2d} done")

print("\n" + "="*70)
print("Pathway-level morphology attribution (pooled across 25 seeds' test sets)")
print("="*70)
pathway_results = {}
for pathway in sorted(pathway_morpho_shap.keys()):
    n_obs = len(pathway_morpho_shap[pathway])
    if n_obs < 4:
        continue
    total_morpho = sum(pathway_morpho_shap[pathway])
    total_struct = sum(pathway_struct_shap[pathway])
    pct_morpho = 100 * total_morpho / (total_morpho + total_struct)
    pathway_results[pathway] = {'pct_morphology': float(pct_morpho), 'n_observations': n_obs}
    print(f"  {pathway:24s} morphology={pct_morpho:5.1f}%  (n={n_obs})")

with open(BASE_DIR / "results" / "shap_pathway_rf_concat.json", "w") as f:
    json.dump(pathway_results, f, indent=2)
print("\nSaved: shap_pathway_rf_concat.json")

Drugs with no pathway match: 0 of 175
[  0.2 min] seed= 0 done
[  1.4 min] seed= 5 done
[  2.6 min] seed=10 done
[  3.9 min] seed=15 done
[  5.2 min] seed=20 done
[  6.3 min] seed=24 done

Pathway-level morphology attribution (pooled across 25 seeds' test sets)
  Apoptosis regulation     morphology= 95.9%  (n=34)
  Cell cycle               morphology= 96.9%  (n=28)
  Chromatin histone acetylation morphology= 95.4%  (n=21)
  Chromatin histone methylation morphology= 95.1%  (n=42)
  Chromatin other          morphology= 94.4%  (n=25)
  DNA replication          morphology= 92.3%  (n=44)
  EGFR signaling           morphology= 94.9%  (n=20)
  ERK MAPK signaling       morphology= 95.2%  (n=40)
  Genome integrity         morphology= 95.2%  (n=37)
  Hormone-related          morphology= 94.4%  (n=20)
  IGF1R signaling          morphology= 95.9%  (n=20)
  Metabolism               morphology= 94.4%  (n=21)
  Mitosis                  morphology= 98.2%  (n=15)
  Other                    morphology= 

In [4]:
# Complete the classical-vs-MLP paired comparisons

with open(BASE_DIR / "results" / "rf_gbm_25seed_all_features.json") as f:
    rf_gbm_full = json.load(f)
with open(BASE_DIR / "results" / "ridgecv_25seed_all_features.json") as f:
    ridge_full = json.load(f)
with open(BASE_DIR / "results" / "multiseed_25_results_drugscored.json") as f:
    mlp_full = json.load(f)

pairs = [
    ('rf_structure_only',    'structure_only',   'RF-structure vs MLP-structure'),
    ('rf_morphology_only',   'morphology_only',  'RF-morphology vs MLP-morphology'),
    ('rf_concatenation',     'concatenation',    'RF-concatenation vs MLP-concatenation'),
    ('gbm_structure_only',   'structure_only',   'GBM-structure vs MLP-structure'),
    ('gbm_morphology_only',  'morphology_only',  'GBM-morphology vs MLP-morphology'),
    ('gbm_concatenation',    'concatenation',    'GBM-concatenation vs MLP-concatenation'),
]

print("Random Forest / Gradient Boosting vs matched MLP condition:")
full_classical_stats = {}
for classical_key, mlp_key, label in pairs:
    classical_arr = np.array(rf_gbm_full[classical_key]['r'])
    mlp_arr = np.array(mlp_full[mlp_key]['r_drug'])
    t, p = ttest_rel(classical_arr, mlp_arr)
    dR = classical_arr.mean() - mlp_arr.mean()
    full_classical_stats[label] = {'t': float(t), 'p': float(p), 'delta_R': float(dR)}
    print(f"  {label:42s} t={t:6.2f}  p={p:.4g}  deltaR={dR:+.3f}")

print("\nRidgeCV vs matched MLP condition:")
ridge_pairs = [
    ('structure_only',  'structure_only',  'RidgeCV-structure vs MLP-structure'),
    ('morphology_only', 'morphology_only', 'RidgeCV-morphology vs MLP-morphology'),
]
for ridge_key, mlp_key, label in ridge_pairs:
    ridge_arr = np.array(ridge_full[ridge_key]['r'])
    mlp_arr = np.array(mlp_full[mlp_key]['r_drug'])
    t, p = ttest_rel(ridge_arr, mlp_arr)
    dR = ridge_arr.mean() - mlp_arr.mean()
    full_classical_stats[label] = {'t': float(t), 'p': float(p), 'delta_R': float(dR)}
    print(f"  {label:42s} t={t:6.2f}  p={p:.4g}  deltaR={dR:+.3f}")

# also RF-concatenation vs cross-attention, and RF-concatenation vs RidgeCV-concatenation,
# already computed previously but re-included here so this cell is the single complete record
rf_concat = np.array(rf_gbm_full['rf_concatenation']['r'])
mlp_ca = np.array(mlp_full['cross_attention']['r_drug'])
t, p = ttest_rel(rf_concat, mlp_ca)
full_classical_stats['RF-concatenation vs best neural (cross-attention)'] = {'t': float(t), 'p': float(p)}
print(f"\n  {'RF-concatenation vs best neural (cross-attention)':42s} t={t:6.2f}  p={p:.4g}")

with open(BASE_DIR / "results" / "classical_vs_mlp_stats_complete.json", "w") as f:
    json.dump(full_classical_stats, f, indent=2)
print("\nSaved: classical_vs_mlp_stats_complete.json")

Random Forest / Gradient Boosting vs matched MLP condition:
  RF-structure vs MLP-structure              t=  2.52  p=0.01863  deltaR=+0.058
  RF-morphology vs MLP-morphology            t=  4.03  p=0.0004905  deltaR=+0.063
  RF-concatenation vs MLP-concatenation      t=  4.36  p=0.0002113  deltaR=+0.056
  GBM-structure vs MLP-structure             t=  0.67  p=0.5085  deltaR=+0.021
  GBM-morphology vs MLP-morphology           t=  2.26  p=0.03311  deltaR=+0.052
  GBM-concatenation vs MLP-concatenation     t=  3.75  p=0.0009864  deltaR=+0.047

RidgeCV vs matched MLP condition:
  RidgeCV-structure vs MLP-structure         t= -0.91  p=0.3721  deltaR=-0.019
  RidgeCV-morphology vs MLP-morphology       t=  2.31  p=0.02952  deltaR=+0.030

  RF-concatenation vs best neural (cross-attention) t=  4.40  p=0.0001899

Saved: classical_vs_mlp_stats_complete.json


In [5]:
from scipy.stats import spearmanr

morgan_raw = pd.read_csv(PROC_DIR / "morgan_fingerprints.csv")
sim_drug_names = morgan_raw['drug_name'].tolist()

tanimoto   = np.load(PROC_DIR / "tanimoto_similarity.npy")
morpho_sim = np.load(PROC_DIR / "morpho_cosine_similarity.npy")

assert tanimoto.shape == (len(sim_drug_names), len(sim_drug_names)), "shape mismatch — wrong file or order"
assert morpho_sim.shape == tanimoto.shape

ic50_matrix = pd.read_csv(PROC_DIR / "gdsc2_ic50_matrix.csv", index_col=0)
assert set(sim_drug_names) == set(ic50_matrix.index), "drug sets don't match — check for typos or missing drugs"
X = ic50_matrix.loc[sim_drug_names].values.astype(float)

def cosine_pairwise_complete(X, min_overlap=10):
    n = X.shape[0]
    sim = np.full((n, n), np.nan)
    mask = ~np.isnan(X)
    for i in range(n):
        for j in range(i, n):
            m = mask[i] & mask[j]
            if m.sum() < min_overlap:
                continue
            a, b = X[i, m], X[j, m]
            sim[i, j] = sim[j, i] = (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)
    np.fill_diagonal(sim, 1.0)
    return sim

ic50_sim = cosine_pairwise_complete(X)
n = len(sim_drug_names)
iu = np.triu_indices(n, k=1)
n_excluded = np.isnan(ic50_sim[iu]).sum()
print(f"Drug pairs excluded (fewer than 10 shared cell lines): {n_excluded}")

def mantel_permutation_test(A, B, iu, n_perm=10000, seed=0):
    valid0 = ~np.isnan(B[iu])
    obs_r, _ = spearmanr(A[iu][valid0], B[iu][valid0])
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    n_obj = A.shape[0]
    for k in range(n_perm):
        perm = rng.permutation(n_obj)
        B_perm = B[np.ix_(perm, perm)]
        valid = ~np.isnan(B_perm[iu])
        r, _ = spearmanr(A[iu][valid], B_perm[iu][valid])
        null[k] = r
    p = (np.sum(np.abs(null) >= np.abs(obs_r)) + 1) / (n_perm + 1)
    return float(obs_r), float(p)

r_orthog, p_orthog       = mantel_permutation_test(tanimoto,   morpho_sim, iu, seed=2)
r_tan_ic50, p_tan_ic50   = mantel_permutation_test(tanimoto,   ic50_sim,   iu, seed=0)
r_morph_ic50, p_morph_ic50 = mantel_permutation_test(morpho_sim, ic50_sim, iu, seed=1)

print(f"\nOrthogonality (chemistry vs morphology):  r={r_orthog:.4f}  permutation p={p_orthog:.4f}")
print(f"Chemistry vs response similarity:         r={r_tan_ic50:.4f}  permutation p={p_tan_ic50:.4f}")
print(f"Morphology vs response similarity:        r={r_morph_ic50:.4f}  permutation p={p_morph_ic50:.4f}")

with open(PROC_DIR / "similarity_corrected.json", "w") as f:
    json.dump({
        'orthogonality':          {'r': r_orthog,     'p_perm': p_orthog},
        'chemistry_vs_response':  {'r': r_tan_ic50,   'p_perm': p_tan_ic50},
        'morphology_vs_response': {'r': r_morph_ic50, 'p_perm': p_morph_ic50},
        'n_pairs_excluded_overlap': int(n_excluded),
        'n_permutations': 10000,
    }, f, indent=2)
print("\nSaved: similarity_corrected.json")

Drug pairs excluded (fewer than 10 shared cell lines): 48

Orthogonality (chemistry vs morphology):  r=0.0307  permutation p=0.0057
Chemistry vs response similarity:         r=-0.0918  permutation p=0.0336
Morphology vs response similarity:        r=0.2531  permutation p=0.0001

Saved: similarity_corrected.json
